# 🏥 Clinical Video Analysis — Treinamento YOLOv8
Treina 4 modelos especializados por contexto clínico:
- **Cirurgia**: detecção de sangramento
- **Consulta**: sinais de dor/desconforto
- **Fisioterapia**: análise de postura e movimento
- **Triagem**: linguagem corporal indicativa de violência

## 1. Instalar Dependências

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'kagglehub', 'ultralytics', 'pyyaml'], check=True)
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 2. Verificar GPU

In [2]:
import torch
DEVICE = 0
if torch.cuda.is_available():
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 0
else:
    print("GPU não encontrada, usando CPU (treinamento mais lento)")
    DEVICE = 'cpu'


GPU disponível: NVIDIA GeForce RTX 3070
VRAM: 8.6 GB


## 3. Baixar Datasets

In [3]:
import kagglehub, os, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from roboflow import Roboflow

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

paths = {}

print("📥 DOWNLOAD DATASETS FEMHEALTH")
print("=" * 55)

# -----------------------------------------------
# 1. LAPAROSCOPIA ROBFLOW → data/cirurgia/
# -----------------------------------------------
cirurgia_path = os.path.join(DATA_DIR, "cirurgia")
if os.path.exists(cirurgia_path):
    print("✅ CIRURGIA já existe, pulando download")
    paths['cirurgia'] = cirurgia_path
else:
    print("🔄 1/4 CIRURGIA: Laparoscopia...")
    rf = Roboflow(api_key="qYZBO5fqNIjOlDK6jH4o")  # SUA chave!
    project = rf.workspace("laparoscopic-yolo").project("laparoscopy")
    dataset_obj = project.version(14).download("yolov8")  # Retorna Dataset obj

    # Extrai path real (fix TypeError)
    temp_path = dataset_obj.location  # ← FIX: .location da Dataset obj
    shutil.move(temp_path, cirurgia_path)
    paths['cirurgia'] = cirurgia_path
    print("✅ CIRURGIA: data/cirurgia/ ✓")

# -----------------------------------------------
# 2. KAGGLE → data/{context}/
# -----------------------------------------------
DATASETS_KAGGLE = {
    'consulta':     'coder98/emotionpain',
    'fisioterapia': 'sulaimanmuhammed/wlu-rehabilitation-posture',
    'triagem':      'simuletic/cctv-aggressive-poses-and-fight-detection-dataset',
}

def download_to_data(context, dataset_id):
    target_path = os.path.join(DATA_DIR, context)
    if os.path.exists(target_path):
        return context, target_path, None  # já existe, pula download
    try:
        temp_path = kagglehub.dataset_download(dataset_id)
        shutil.move(temp_path, target_path)
        return context, target_path, None
    except Exception as e:
        return context, None, str(e)

print("\n🔄 2-4/4 KAGGLE...")
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(download_to_data, ctx, ds): ctx for ctx, ds in DATASETS_KAGGLE.items()}
    for future in as_completed(futures):
        context, path, error = future.result()
        if error:
            print(f"❌ {context:>11}: {error}")
        else:
            paths[context] = path
            print(f"✅ {context:>11}: OK")

# -----------------------------------------------
# Estrutura final
# -----------------------------------------------
print("\n✅ ESTRUTURA:")
print(f"data/")
for ctx in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    status = "✅" if ctx in paths else "❌"
    print(f"├── {ctx}/  {status}")

📥 DOWNLOAD DATASETS FEMHEALTH
✅ CIRURGIA já existe, pulando download

🔄 2-4/4 KAGGLE...
✅    consulta: OK
✅ fisioterapia: OK
✅     triagem: OK

✅ ESTRUTURA:
data/
├── cirurgia/  ✅
├── consulta/  ✅
├── fisioterapia/  ✅
├── triagem/  ✅


## 4. Inspecionar Estrutura dos Datasets

In [4]:
import os

def inspect_dataset(path, name):
    """Status claro COM emojis explicados"""
    if not os.path.exists(path):
        return f"{name:12} ❌ Pasta inexistente"
    
    imgs = sum(1 for root, _, files in os.walk(path) 
               for f in files if f.lower().endswith(('.jpg','.jpeg','.png')))
    
    labels = sum(1 for root, _, files in os.walk(path) 
                 for f in files if f.endswith('.txt'))
    
    videos = sum(1 for root, _, files in os.walk(path) 
                 for f in files if f.lower().endswith(('.mp4','.avi')))
    
    yaml_ok = os.path.exists(os.path.join(path, 'data.yaml'))
    
    # STATUS COM EXPLICAÇÃO
    if yaml_ok and imgs > 0 and labels > 0 and abs(imgs-labels) < 20:
        status = "✅ YOLOv8 pronto (tem yaml+imgs+labels)"
    elif videos > 0:
        status = "🎥 Vídeos crus (precisa extrair frames)" 
    elif imgs > 0 and labels == 0:
        status = "🖼️ Apenas imagens (classification ou rotular)"
    elif imgs == 0 and labels == 0:
        status = "📂 Vazio (precisa download/preprocess)"
    else:
        status = "⚠️ Inconsistente (imgs≠labels)"
    
    return f"{name:12} | {imgs:>4}i {labels:>3}l {videos:>3}v | {status}"

print("📊 DATASETS FEMHEALTH")
print("="*70)
print("i=imagens l=labels v=vídeos")
print("-"*70)

for ctx, path in paths.items():
    print(inspect_dataset(path, ctx.upper()))


📊 DATASETS FEMHEALTH
i=imagens l=labels v=vídeos
----------------------------------------------------------------------
CIRURGIA     | 2338i 2342l   0v | ✅ YOLOv8 pronto (tem yaml+imgs+labels)
CONSULTA     | 48398i 96796l   0v | ⚠️ Inconsistente (imgs≠labels)
FISIOTERAPIA |    0i   0l 514v | 🎥 Vídeos crus (precisa extrair frames)
TRIAGEM      |  206i 206l   0v | ⚠️ Inconsistente (imgs≠labels)


In [6]:
import os, yaml
from pathlib import Path

def analyze_dataset(path):
    """Análise YOLOv8 completa e precisa"""
    p = Path(path)
    if not p.exists():
        return {'error': 'Pasta não existe'}
    
    stats = {
        'videos': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.mp4','.mov','.avi'}),
        'images': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.jpg','.jpeg','.png'}),
        'labels_txt': sum(1 for x in p.rglob('*.txt')),
        'yaml': False,
        'train_split': bool(p / 'train'),
        'valid_split': bool(p / 'val'),
        'classes_yaml': [],
        'train_path_ok': False,
        'val_path_ok': False
    }
    
    # YAML análise detalhada
    yaml_file = next(p.rglob('data.yaml'), None)
    if yaml_file and yaml_file.exists():
        stats['yaml'] = True
        try:
            with open(yaml_file) as f:
                data = yaml.safe_load(f)
                stats['classes_yaml'] = data.get('names', [])
                train_rel = data.get('train', '')
                val_rel = data.get('val', '')
                stats['train_path_ok'] = (p / train_rel).exists()
                stats['val_path_ok'] = (p / val_rel).exists()
        except:
            stats['yaml'] = False
    
    stats['imgs_labels_ratio'] = 'OK' if abs(stats['images'] - stats['labels_txt']) < 20 else f"{stats['images']}/{stats['labels_txt']}"
    return stats

# ════════════════════════════════════════════════════════════════
print("🔍 VERIFICAÇÃO YOLOv8 DETALHADA")
print("="*75)
print("imgs | lbls | yaml | train/val | classes | status")
print("-"*75)

for context, path in paths.items():
    stats = analyze_dataset(path)
    
    if 'error' in stats:
        print(f"{context.upper():<12} | ❌ {stats['error']}")
        continue
    
    # Linha única clara
    yaml_status = "✅" if stats['yaml'] else "❌"
    train_status = "✅" if stats['train_split'] and stats['train_path_ok'] else "❌"
    val_status = "✅" if stats['valid_split'] and stats['val_path_ok'] else "❌"
    
    ready = stats['yaml'] and stats['images'] > 0 and stats['labels_txt'] > 0 and stats['train_path_ok'] and stats['val_path_ok']
    status = "🚀 PRONTO TREINO" if ready else "⚠️ PROBLEMAS"
    
    classes_preview = str(stats['classes_yaml'])[:30] + "..." if stats['classes_yaml'] else "N/A"
    
    print(f"{context.upper():<12} | {stats['images']:>4}/{stats['labels_txt']:>4} | {yaml_status} | {train_status}/{val_status} | {classes_preview:<30} | {status}")

print("\n" + "="*75)
print("🎯 COMANDOS TREINO PRONTOS:")
for ctx in ['cirurgia', 'triagem']:
    if ctx in paths:
        stats = analyze_dataset(paths[ctx])
        if stats.get('yaml') and stats['train_path_ok']:
            print(f"  YOLO('yolov8n.pt').train(data='{paths[ctx]}', epochs=50)")
        else:
            print(f"  {ctx.upper()}: ⚠️ PREPROCESS primeiro")


🔍 VERIFICAÇÃO YOLOv8 DETALHADA
imgs | lbls | yaml | train/val | classes | status
---------------------------------------------------------------------------
CIRURGIA     | 2338/2342 | ✅ | ❌/❌ | ['Allis', 'Bag', 'Cautery', 'F... | ⚠️ PROBLEMAS
CONSULTA     | 48398/96796 | ❌ | ❌/❌ | N/A                            | ⚠️ PROBLEMAS
FISIOTERAPIA |    0/   0 | ❌ | ❌/❌ | N/A                            | ⚠️ PROBLEMAS
TRIAGEM      |  206/ 206 | ✅ | ❌/❌ | {0: 'person'}...               | ⚠️ PROBLEMAS

🎯 COMANDOS TREINO PRONTOS:
  CIRURGIA: ⚠️ PREPROCESS primeiro
  TRIAGEM: ⚠️ PREPROCESS primeiro


## 5. Gerar data.yaml para Cada Contexto

In [5]:
import os, cv2, yaml, random, shutil
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

DATASET_OUT = Path("datasets/yolov8")
DATASET_OUT.mkdir(parents=True, exist_ok=True)

def safe_mkdir(p):
    p.mkdir(parents=True, exist_ok=True)

def extract_frames(video_path, out_dir, step=5):
    cap = cv2.VideoCapture(str(video_path))
    frame_id = saved = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_id % step == 0:
            out = out_dir / f"{video_path.stem}_{frame_id:06d}.jpg"
            cv2.imwrite(str(out), frame)
            saved += 1
        frame_id += 1
    cap.release()
    return saved

def split_list(items, train_ratio=0.8):
    random.shuffle(items)
    n = int(len(items) * train_ratio)
    return items[:n], items[n:]

def _find_label(dataset_root: Path, img_path: Path) -> Path | None:
    # 1. Mesmo diretório
    same_dir = img_path.with_suffix(".txt")
    if same_dir.exists():
        return same_dir
    # 2. Substitui 'images' → 'labels'
    str_path = str(img_path)
    if "images" in str_path:
        lbl_path = Path(str_path.replace("images", "labels")).with_suffix(".txt")
        if lbl_path.exists():
            return lbl_path
    # 3. Busca por stem na pasta labels/ do dataset_root
    for candidate in (dataset_root / "labels").rglob(f"{img_path.stem}.txt"):
        return candidate
    return None

def _corrigir_label(lbl_path: Path):
    """
    Corrige labels onde o primeiro campo é a classe como float (ex: 0.000000).
    Substitui por int em vez de adicionar novo campo.
    """
    linhas   = open(lbl_path).readlines()
    novas    = []
    alterado = False
    for linha in linhas:
        linha = linha.strip()
        if not linha:
            continue
        parts = linha.split()
        try:
            int(parts[0])          # já é int → ok
            novas.append(linha + "\n")
        except ValueError:
            try:
                cls = int(float(parts[0]))   # era float → converte para int
                novas.append(f"{cls} " + " ".join(parts[1:]) + "\n")
                alterado = True
            except ValueError:
                novas.append(linha + "\n")
    if alterado:
        with open(lbl_path, "w") as f:
            f.writelines(novas)



# ── CIRURGIA ──────────────────────────────────────────────────────────────
def process_cirurgia(path):
    print("🔧 CIRURGIA: Laparoscopia Roboflow")
    src = Path(path)
    out = DATASET_OUT / "cirurgia"
    safe_mkdir(out)
    shutil.copytree(src, out, dirs_exist_ok=True)

    train_dir = next((p for p in [out/"train/images", out/"train"] if p.exists()), None)
    val_dir   = next((p for p in [out/"valid/images", out/"val/images", out/"valid", out/"val"] if p.exists()), None)

    yaml_path = out / "data.yaml"
    if yaml_path.exists():
        with open(yaml_path) as f:
            data = yaml.safe_load(f)
        classes = data.get("names", [])
    else:
        classes = []

    new_yaml = {
        "path":  str(out.resolve()),
        "train": str(train_dir.resolve()) if train_dir else "train/images",
        "val":   str(val_dir.resolve())   if val_dir   else "valid/images",
        "nc":    len(classes),
        "names": classes,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(new_yaml, f)

    print(f"✅ Classes: {classes}")
    print(f"   train → {new_yaml['train']}")
    print(f"   val   → {new_yaml['val']}")
    return "cirurgia", str(yaml_path)


# ── TRIAGEM ───────────────────────────────────────────────────────────────
def process_triagem(path):
    print("🔧 TRIAGEM: Aggressive poses")
    src = Path(path)
    out = DATASET_OUT / "triagem"
    safe_mkdir(out)

    dataset_root = next((p for p in [src/"Aggressive_Poses_Dataset", src]
                         if (p/"images").exists() or list(p.glob("**/*.jpg"))), src)

    raw_out = out / "raw"
    safe_mkdir(raw_out)
    shutil.copytree(dataset_root, raw_out, dirs_exist_ok=True)

    # Apenas imagens da pasta images/ (não subpastas train/val já criadas)
    all_images     = list((raw_out / "images").glob("*.jpg")) + \
                     list((raw_out / "images").glob("*.png")) \
                     if (raw_out / "images").exists() else \
                     list(raw_out.rglob("*.jpg")) + list(raw_out.rglob("*.png"))

    train_imgs_dir = raw_out / "images" / "train"
    val_imgs_dir   = raw_out / "images" / "val"
    train_lbls_dir = raw_out / "labels" / "train"
    val_lbls_dir   = raw_out / "labels" / "val"

    if not train_imgs_dir.exists():
        print(f"  📂 Criando split train/val de {len(all_images)} imagens...")
        for d in [train_imgs_dir, val_imgs_dir, train_lbls_dir, val_lbls_dir]:
            safe_mkdir(d)

        train_list, val_list = split_list(all_images)
        n_lbl_train = n_lbl_val = 0

        for img in train_list:
            shutil.copy(img, train_imgs_dir / img.name)
            lbl = _find_label(raw_out, img)
            if lbl and lbl.exists():
                dest = train_lbls_dir / lbl.name
                shutil.copy(lbl, dest)
                _corrigir_label(dest)
                n_lbl_train += 1

        for img in val_list:
            shutil.copy(img, val_imgs_dir / img.name)
            lbl = _find_label(raw_out, img)
            if lbl and lbl.exists():
                dest = val_lbls_dir / lbl.name
                shutil.copy(lbl, dest)
                _corrigir_label(dest)
                n_lbl_val += 1

        print(f"  ✅ train: {len(train_list)} imgs | {n_lbl_train} labels")
        print(f"  ✅ val:   {len(val_list)} imgs | {n_lbl_val} labels")

    # Detecta kpt_shape — dataset é COCO pose, força 17 × 3
    n_kpts  = 17
    kpt_dim = 3
    for lbl_file in list(train_lbls_dir.glob("*.txt"))[:5]:
        for line in open(lbl_file).readlines():
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            n_coords = len(parts) - 5  # desconta classe(1) + bbox(4)
            # Prioriza dim=3 (com visibility) sobre dim=2
            if n_coords % 3 == 0:
                n_kpts  = n_coords // 3
                kpt_dim = 3
                break
            elif n_coords % 2 == 0:
                # Pode ser 17×2=34 ou artefato — só aceita se n_kpts conhecido
                candidate = n_coords // 2
                if candidate in (17, 13, 21):  # valores conhecidos de pose
                    n_kpts  = candidate
                    kpt_dim = 2
                    break
        break

    print(f"  🦴 Keypoints detectados: {n_kpts} × {kpt_dim} valores")

    all_labels  = list(train_lbls_dir.glob("*.txt")) + list(val_lbls_dir.glob("*.txt"))
    classes_ids = sorted(set(
        int(float(l.split()[0]))
        for lbl in all_labels
        for l in open(lbl) if l.strip()
    )) if all_labels else [0]

    data_yaml = {
        "path":      str(out.resolve()),
        "train":     str(train_imgs_dir.resolve()),
        "val":       str(val_imgs_dir.resolve()),
        "nc":        len(classes_ids),
        "names":     {i: f"person_pose_{i}" for i in classes_ids},
        "kpt_shape": [n_kpts, kpt_dim],
    }
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    print(f"  ✅ kpt_shape: [{n_kpts}, {kpt_dim}]")
    return "triagem", str(yaml_path)


# ── FISIOTERAPIA ──────────────────────────────────────────────────────────
def process_fisioterapia(path):
    print("🔧 FISIOTERAPIA: Vídeos → frames")
    src = Path(path)
    out = DATASET_OUT / "fisioterapia"
    safe_mkdir(out)

    all_videos = list(src.rglob("*.mp4")) + list(src.rglob("*.avi")) + list(src.rglob("*.mov"))
    videos_por_tema = defaultdict(list)
    for v in all_videos:
        videos_por_tema[v.parent.name].append(v)

    print(f"🎥 {len(all_videos)} vídeos totais | {len(videos_por_tema)} temas encontrados:")

    MAX_VIDEOS_POR_TEMA = 20
    videos_selecionados = []
    stem_to_tema = {}
    for tema, vids in videos_por_tema.items():
        selecionados = random.sample(vids, min(MAX_VIDEOS_POR_TEMA, len(vids)))
        print(f"   📁 {tema}: {len(vids)} disponíveis → {len(selecionados)} selecionados")
        for v in selecionados:
            stem_to_tema[v.stem] = tema
        videos_selecionados.extend(selecionados)

    print(f"🎬 Total selecionado: {len(videos_selecionados)} vídeos")

    if not videos_selecionados:
        print("⚠️ Sem vídeos encontrados")
        return "fisioterapia", str(out)

    temp_dir = out / "temp_frames"
    safe_mkdir(temp_dir)

    total_frames = 0
    with ThreadPoolExecutor(max_workers=2) as ex:
        futures = {ex.submit(extract_frames, v, temp_dir): v for v in videos_selecionados}
        for f in futures:
            total_frames += f.result()
    print(f"🖼️ {total_frames} frames extraídos")

    frames_por_tema = defaultdict(list)
    for frame in temp_dir.glob("*.jpg"):
        video_stem = "_".join(frame.stem.split("_")[:-1])
        tema = stem_to_tema.get(video_stem, "desconhecido")
        frames_por_tema[tema].append(frame)

    print(f"  📂 {len(frames_por_tema)} classes encontradas:")
    for tema, frames in frames_por_tema.items():
        print(f"     {tema}: {len(frames)} frames")

    for tema, frames in frames_por_tema.items():
        cls_name = tema.replace(" ", "_")
        train_frames, val_frames = split_list(frames)
        safe_mkdir(out / "train" / cls_name)
        safe_mkdir(out / "val"   / cls_name)
        for f in train_frames:
            shutil.move(str(f), out / "train" / cls_name / f.name)
        for f in val_frames:
            shutil.move(str(f), out / "val"   / cls_name / f.name)

    shutil.rmtree(temp_dir, ignore_errors=True)

    class_names = sorted([d.name for d in (out / "train").iterdir() if d.is_dir()])
    data_yaml = {
        "path":  str(out.resolve()),
        "train": str((out / "train").resolve()),
        "val":   str((out / "val").resolve()),
        "nc":    len(class_names),
        "names": class_names,
    }
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    return "fisioterapia", str(yaml_path)


# ── CONSULTA ──────────────────────────────────────────────────────────────
def process_consulta(path):
    print("🔧 CONSULTA: UNBC-McMaster Pain (FACS → classify)")
    src = Path(path)
    out = DATASET_OUT / "consulta"
    safe_mkdir(out)

    print("  🔢 Lendo scores FACS...")
    img_label_map = {}

    for facs_file in src.rglob("*_facs.txt"):
        stem_img = facs_file.stem.replace("_facs", "")
        conteudo = facs_file.read_text().strip()
        if not conteudo:
            if stem_img not in img_label_map:
                img_label_map[stem_img] = 0
            continue
        try:
            valores = [float(v) for v in conteudo.split()]
            score   = sum(valores[1:]) if len(valores) > 1 else 0.0
        except ValueError:
            score = 0.0
        img_label_map[stem_img] = max(img_label_map.get(stem_img, 0), 1 if score > 0 else 0)

    n_dor     = sum(1 for c in img_label_map.values() if c == 1)
    n_sem_dor = sum(1 for c in img_label_map.values() if c == 0)
    print(f"  😣 com dor:  {n_dor}")
    print(f"  😊 sem dor:  {n_sem_dor}")

    all_imgs = list(src.rglob("*.png"))
    pares    = [(img, img_label_map[img.stem]) for img in all_imgs if img.stem in img_label_map]
    print(f"  🔗 {len(pares)} imagens com label associado")

    if not pares:
        print("  ⚠️ Nenhuma imagem associada — abortando")
        return "consulta", str(out)

    MAX_POR_CLASSE = 2000
    pares_dor     = [(p, c) for p, c in pares if c == 1]
    pares_sem_dor = [(p, c) for p, c in pares if c == 0]
    random.shuffle(pares_dor)
    random.shuffle(pares_sem_dor)
    pares_balanceados = pares_dor[:MAX_POR_CLASSE] + pares_sem_dor[:MAX_POR_CLASSE]
    random.shuffle(pares_balanceados)
    print(f"  ⚖️  Amostra balanceada: {len(pares_balanceados)} ({len(pares_dor[:MAX_POR_CLASSE])} dor + {len(pares_sem_dor[:MAX_POR_CLASSE])} sem dor)")

    class_names = ["sem_dor", "com_dor"]
    for split in ["train", "val"]:
        for cls in class_names:
            safe_mkdir(out / split / cls)

    train_list, val_list = split_list(pares_balanceados, train_ratio=0.8)
    for img_path, classe in train_list:
        shutil.copy(img_path, out / "train" / class_names[classe] / img_path.name)
    for img_path, classe in val_list:
        shutil.copy(img_path, out / "val"   / class_names[classe] / img_path.name)

    print(f"  ✅ train: {len(train_list)} | val: {len(val_list)}")

    data_yaml = {
        "path":  str(out.resolve()),
        "train": str((out / "train").resolve()),
        "val":   str((out / "val").resolve()),
        "nc":    2,
        "names": class_names,
    }
    yaml_path = out / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    return "consulta", str(yaml_path)


# ── EXECUTA ───────────────────────────────────────────────────────────────
tasks = {
    "cirurgia":     process_cirurgia,
    "triagem":      process_triagem,
    "fisioterapia": process_fisioterapia,
    "consulta":     process_consulta,
}

yamls = {}
print("🚀 PREPROCESSAMENTO...")
for name, func in tasks.items():
    if name in paths:
        try:
            result = func(paths[name])
            yamls[result[0]] = result[1]
            print(f"✅ {name}")
        except Exception as e:
            import traceback
            print(f"❌ {name}: {e}")
            traceback.print_exc()

print("\n📄 YAMLs:", yamls)


🚀 PREPROCESSAMENTO...
🔧 CIRURGIA: Laparoscopia Roboflow
✅ Classes: ['Allis', 'Bag', 'Cautery', 'Forceps', 'Gallbladder', 'Liver', 'Suction']
   train → C:\Users\Xuanzang\dev\tech-challeng-fase-4\notebooks\datasets\yolov8\cirurgia\train\images
   val   → C:\Users\Xuanzang\dev\tech-challeng-fase-4\notebooks\datasets\yolov8\cirurgia\valid\images
✅ cirurgia
🔧 TRIAGEM: Aggressive poses
  📂 Criando split train/val de 103 imagens...
  ✅ train: 82 imgs | 82 labels
  ✅ val:   21 imgs | 21 labels
  🦴 Keypoints detectados: 17 × 3 valores
  ✅ kpt_shape: [17, 3]
✅ triagem
🔧 FISIOTERAPIA: Vídeos → frames
🎥 678 vídeos totais | 6 temas encontrados:
   📁 Arm Raise Correct: 102 disponíveis → 20 selecionados
   📁 Arm Raise Incorrect: 110 disponíveis → 20 selecionados
   📁 Knee Extension Correct: 88 disponíveis → 20 selecionados
   📁 Knee Extension Incorrect: 124 disponíveis → 20 selecionados
   📁 Sit To Stand Correct: 158 disponíveis → 20 selecionados
   📁 Sit To Stand Incorrect: 96 disponíveis → 20 sele

In [6]:
import yaml
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
POSE_KP    = 17

TASK_MAP = {
    "cirurgia":     "segment",
    "triagem":      "pose",
    "fisioterapia": "classify",
    "consulta":     "classify",
}

def fix_path(p: str) -> Path:
    return Path(str(p).replace("\\\\", "\\"))

def resolve(root: Path, rel: str) -> Path | None:
    if not rel:
        return None
    p = Path(rel)
    if p.is_absolute():
        return p if p.exists() else None
    for candidate in [(root / p).resolve(), (root.parent / p).resolve()]:
        if candidate.exists():
            return candidate
    return None

def count_files(path: Path, exts) -> int:
    return sum(1 for f in path.rglob("*") if f.suffix.lower() in exts)

def check_labels_paired(img_dir: Path, lbl_dir: Path):
    imgs   = {f.stem for f in img_dir.rglob("*") if f.suffix.lower() in IMAGE_EXTS}
    labels = {f.stem for f in lbl_dir.rglob("*.txt")}
    sem_label = imgs - labels
    sem_img   = labels - imgs
    erros = list(sem_label)[:3] + [f"[orphan] {x}" for x in list(sem_img)[:3]]
    return len(sem_label), len(sem_img), erros

def check_label_format(lbl_dir: Path, nc: int, task: str = "detect") -> list:
    erros = []
    for lbl_file in list(lbl_dir.rglob("*.txt"))[:50]:
        for i, line in enumerate(open(lbl_file).readlines()):
            line = line.strip()
            if not line:
                continue
            parts = line.split()

            try:
                cls = int(float(parts[0]))
            except (ValueError, IndexError):
                erros.append(f"{lbl_file.name}:{i+1} → classe não é inteiro")
                break

            if cls < 0 or cls >= nc:
                erros.append(f"{lbl_file.name}:{i+1} → classe {cls} fora de [0,{nc-1}]")
                break

            coords = parts[1:]

            if task == "detect":
                if len(parts) != 5:
                    erros.append(f"{lbl_file.name}:{i+1} → {len(parts)} campos (esperado 5)")
                    break
                try:
                    vals = [float(v) for v in coords]
                except ValueError:
                    erros.append(f"{lbl_file.name}:{i+1} → valores não numéricos")
                    break
                if not all(0.0 <= v <= 1.0 for v in vals):
                    erros.append(f"{lbl_file.name}:{i+1} → coords fora de [0,1]: {vals}")
                    break

            elif task == "segment":
                if len(parts) < 7:
                    erros.append(f"{lbl_file.name}:{i+1} → poucos campos para segmentação: {len(parts)}")
                    break
                if len(coords) % 2 != 0:
                    erros.append(f"{lbl_file.name}:{i+1} → coords devem ser pares (x,y): {len(coords)}")
                    break
                try:
                    vals = [float(v) for v in coords]
                except ValueError:
                    erros.append(f"{lbl_file.name}:{i+1} → valores não numéricos")
                    break
                fora = [v for v in vals if not (0.0 <= v <= 1.0)]
                if fora:
                    erros.append(f"{lbl_file.name}:{i+1} → {len(fora)} coord(s) fora de [0,1]")
                    break

            elif task == "pose":
                expected_vis   = 5 + POSE_KP * 3  # 56
                expected_novis = 5 + POSE_KP * 2  # 39
                if len(parts) not in (expected_vis, expected_novis):
                    erros.append(
                        f"{lbl_file.name}:{i+1} → {len(parts)} campos "
                        f"(esperado {expected_vis} com visibility ou {expected_novis} sem)"
                    )
                    break
                try:
                    bbox = [float(v) for v in coords[:4]]
                except ValueError:
                    erros.append(f"{lbl_file.name}:{i+1} → bbox não numérica")
                    break
                if not all(0.0 <= v <= 1.0 for v in bbox):
                    erros.append(f"{lbl_file.name}:{i+1} → bbox fora de [0,1]: {bbox}")
                    break

        if len(erros) >= 5:
            break
    return erros


def verificar_dataset_yolov8(name: str, yaml_path_raw: str, task: str) -> bool:
    print(f"\n{'='*65}")
    print(f"🔍 {name.upper()} — task: {task}")
    print(f"{'='*65}")

    yaml_path = fix_path(yaml_path_raw)
    erros  = []
    avisos = []

    # 1. YAML legível
    if not yaml_path.exists():
        print(f"  ❌ data.yaml não encontrado: {yaml_path}")
        return False
    try:
        with open(yaml_path) as f:
            data = yaml.safe_load(f)
    except Exception as e:
        print(f"  ❌ Erro ao ler YAML: {e}")
        return False
    print(f"  ✅ data.yaml carregado")

    # 2. Campos obrigatórios
    for campo in ["train", "val", "nc", "names"]:
        if campo not in data:
            erros.append(f"campo '{campo}' ausente no YAML")

    nc    = data.get("nc", 0)
    names = data.get("names", [])
    if isinstance(names, dict):
        names = list(names.values())
    n_names = len(names)
    nc_ok   = nc == n_names
    print(f"  {'✅' if nc_ok else '⚠️ '} nc={nc} | classes={n_names} | {names[:5]}{'...' if n_names > 5 else ''}")
    if not nc_ok:
        erros.append(f"nc={nc} ≠ len(names)={n_names}")

    # 2b. kpt_shape obrigatório para pose
    if task == "pose":
        kpt_shape = data.get("kpt_shape")
        if not kpt_shape:
            erros.append("campo 'kpt_shape' ausente no YAML (obrigatório para pose)")
            print(f"  ❌ kpt_shape: ausente")
        elif not (isinstance(kpt_shape, list) and len(kpt_shape) == 2):
            erros.append(f"kpt_shape inválido: {kpt_shape} (esperado [n_kpts, dim])")
            print(f"  ❌ kpt_shape: {kpt_shape} inválido")
        else:
            nk, dk = kpt_shape
            print(f"  ✅ kpt_shape: [{nk}, {dk}] ({nk} keypoints × {dk} valores)")

    root = yaml_path.resolve().parent

    # 3. Splits
    if task in ("detect", "segment", "pose"):
        for split in ["train", "val"]:
            raw_split = data.get(split, "")
            img_dir   = resolve(root, raw_split)
            if img_dir is None and split == "val":
                img_dir = resolve(root, raw_split.replace("val", "valid"))

            if img_dir is None or not img_dir.exists():
                erros.append(f"split '{split}' não encontrado: {raw_split}")
                print(f"  ❌ {split}: pasta não existe → {raw_split}")
                continue

            lbl_dir  = Path(str(img_dir).replace("images", "labels"))
            n_imgs   = count_files(img_dir, IMAGE_EXTS)
            n_labels = count_files(lbl_dir, {".txt"}) if lbl_dir.exists() else 0
            ratio_ok = n_labels > 0 and abs(n_imgs - n_labels) <= max(5, int(n_imgs * 0.05))

            try:
                display_path = img_dir.resolve().relative_to(root)
            except ValueError:
                display_path = img_dir

            print(f"  {'✅' if ratio_ok else '⚠️ '} {split}: {n_imgs} imgs | {n_labels} labels | {display_path}")

            if n_imgs == 0:
                erros.append(f"split '{split}' sem imagens")
            if not lbl_dir.exists():
                erros.append(f"pasta labels/{split} não encontrada")
                print(f"  ❌ labels/{split}: {lbl_dir} não existe")
                continue

            sem_lbl, sem_img, _ = check_labels_paired(img_dir, lbl_dir)
            if sem_lbl > 0:
                avisos.append(f"{split}: {sem_lbl} imagem(ns) sem label")
            if sem_img > 0:
                avisos.append(f"{split}: {sem_img} label(s) sem imagem")

            fmt_erros = check_label_format(lbl_dir, nc, task)
            if fmt_erros:
                for e in fmt_erros:
                    erros.append(f"[{split}] {e}")
                print(f"  ❌ labels/{split}: {len(fmt_erros)} erro(s) de formato")
            else:
                tag = {
                    "detect":  "class x y w h ∈ [0,1]",
                    "segment": "polígonos (x,y) ∈ [0,1]",
                    "pose":    f"bbox + {data.get('kpt_shape', [POSE_KP, 3])[0]} keypoints",
                }[task]
                print(f"  ✅ labels/{split}: formato válido ({tag})")

    elif task == "classify":
        for split in ["train", "val"]:
            raw_split = data.get(split, "")
            split_dir = fix_path(raw_split) if Path(raw_split).is_absolute() else root / split
            if not split_dir.exists():
                erros.append(f"split '{split}' não encontrado: {split_dir}")
                print(f"  ❌ {split}: {split_dir} não existe")
                continue
            classes   = [d for d in split_dir.iterdir() if d.is_dir()]
            total     = sum(count_files(c, IMAGE_EXTS) for c in classes)
            cls_names = [c.name for c in classes]
            print(f"  {'✅' if total > 0 else '❌'} {split}: {total} imgs | {len(classes)} classes: {cls_names[:5]}{'...' if len(cls_names) > 5 else ''}")
            if total == 0:
                erros.append(f"split '{split}' sem imagens")
            if not classes:
                erros.append(f"split '{split}' sem subpastas de classe")

    # 4. Resultado
    print()
    for a in avisos:
        print(f"  ⚠️  {a}")
    if erros:
        print(f"  ❌ REPROVADO — {len(erros)} erro(s):")
        for e in erros:
            print(f"     • {e}")
        return False
    print(f"  🚀 APROVADO — pronto para YOLOv8 ({task})")
    return True


# ── EXECUTA ───────────────────────────────────────────────────────────────
print("🧪 VERIFICAÇÃO DE CONFORMIDADE YOLOv8")
print("=" * 65)

aprovados = {}
for name, yp in yamls.items():
    task = TASK_MAP.get(name, "detect")
    aprovados[name] = verificar_dataset_yolov8(name, yp, task)

print(f"\n{'='*65}")
print("📊 RESUMO")
print(f"{'='*65}")
for name, ok in aprovados.items():
    print(f"  {name:<15} → {'🚀 PRONTO' if ok else '❌ CORRIGIR'}")

bloqueados = [n for n, ok in aprovados.items() if not ok]
if bloqueados:
    print(f"\n⛔ Corrigir antes do passo 6: {bloqueados}")
else:
    print("\n✅ Todos aprovados! Pode rodar o Passo 6.")


🧪 VERIFICAÇÃO DE CONFORMIDADE YOLOv8

🔍 CIRURGIA — task: segment
  ✅ data.yaml carregado
  ✅ nc=7 | classes=7 | ['Allis', 'Bag', 'Cautery', 'Forceps', 'Gallbladder']...
  ✅ train: 948 imgs | 948 labels | train\images
  ✅ labels/train: formato válido (polígonos (x,y) ∈ [0,1])
  ✅ val: 133 imgs | 133 labels | valid\images
  ✅ labels/val: formato válido (polígonos (x,y) ∈ [0,1])

  🚀 APROVADO — pronto para YOLOv8 (segment)

🔍 TRIAGEM — task: pose
  ✅ data.yaml carregado
  ✅ nc=1 | classes=1 | ['person_pose_0']
  ✅ kpt_shape: [17, 3] (17 keypoints × 3 valores)
  ✅ train: 82 imgs | 82 labels | raw\images\train
  ✅ labels/train: formato válido (bbox + 17 keypoints)
  ✅ val: 21 imgs | 21 labels | raw\images\val
  ✅ labels/val: formato válido (bbox + 17 keypoints)

  🚀 APROVADO — pronto para YOLOv8 (pose)

🔍 FISIOTERAPIA — task: classify
  ✅ data.yaml carregado
  ✅ nc=6 | classes=6 | ['Arm_Raise_Correct', 'Arm_Raise_Incorrect', 'Knee_Extension_Correct', 'Knee_Extension_Incorrect', 'Sit_To_Stan

## 6. Treinar os Modelos

In [9]:
from ultralytics import YOLO
import traceback, yaml
from pathlib import Path

DEVICE   = 0
BASE_DIR = Path("models/yolov8_femhealth")
BASE_DIR.mkdir(parents=True, exist_ok=True)

resultados = {}
status     = {}

def safe_train(name, base_model, data_path, task="detect", epochs=30):
    expected_dir = BASE_DIR / task / name
    best_pt      = expected_dir / "weights" / "best.pt"
    if best_pt.exists():
        print(f"✅ {name.upper()} SKIP — já treinado em {best_pt}")
        resultados[name] = str(expected_dir)
        return "SKIP"
    
    if task == "classify":
        data_path = str(Path(data_path).parent)

    print(f"\n🚀 TREINO: {name.upper()} ({task})")
    print(f"📁 data: {data_path}")

    try:
        params = {
            "data":        data_path,
            "epochs":      epochs,
            "imgsz":       640 if task != "classify" else 224,
            "batch":       16  if task != "classify" else 32,
            "device":      DEVICE,
            "project":     str(BASE_DIR / task),   # ← project = BASE_DIR/task
            "name":        name,                   # ← name = só o nome
            "exist_ok":    True,
            "patience":    8,
            "workers":     0,
            "save_period": 10,
            "task":        task,
            "augment":     True,
            "val":         True,
        }

        model   = YOLO(base_model)
        results = model.train(**params)
        save_dir = Path(results.save_dir)

        # Verifica se best.pt foi salvo
        best_pt = save_dir / "weights" / "best.pt"
        if best_pt.exists():
            print(f"✅ {name} → {best_pt}")
            resultados[name] = str(save_dir)
            return "OK"
        else:
            # Tenta last.pt como fallback
            last_pt = save_dir / "weights" / "last.pt"
            if last_pt.exists():
                print(f"⚠️  {name} → best.pt não encontrado, usando last.pt")
                resultados[name] = str(save_dir)
                return "OK_LAST"
            else:
                print(f"❌ {name} → nenhum .pt encontrado em {save_dir / 'weights'}")
                print(f"   arquivos em save_dir: {list(save_dir.rglob('*.pt'))}")
                return "ERROR_NO_PT"

    except Exception as e:
        print(f"❌ {name}: {e}")
        traceback.print_exc()
        return "ERROR"


def contar_imagens(yaml_path_raw: str) -> int:
    try:
        yp = Path(str(yaml_path_raw).replace("\\\\", "\\")).resolve()
        with open(yp) as f:
            data = yaml.safe_load(f)
        train_rel = data.get("train", "")
        train_dir = Path(train_rel) if Path(train_rel).is_absolute() else (yp.parent / train_rel).resolve()
        if train_dir.exists():
            return sum(1 for f in train_dir.rglob("*")
                       if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
    except Exception:
        pass
    return 0


# ── Mapa: context → (nome, modelo_base, task, epochs) ────────────────────
# task correta baseada nos labels reais de cada dataset:
#   cirurgia   → segment  (labels são polígonos de segmentação)
#   triagem    → pose     (labels são keypoints, 56 campos)
#   fisioterapia → classify
TRAIN_MAP = {
    "triagem":      ("aggressive_poses", "yolov8n-pose.pt", "pose",     30),
    "consulta":     ("pain_detection",   "yolov8n-cls.pt",  "classify", 30),
    "fisioterapia": ("rehab_postures",   "yolov8n-cls.pt",  "classify", 30),
    "cirurgia":     ("laparoscopia",     "yolov8n-seg.pt",  "segment",  50),
}

# ── Ordena do menor ao maior dataset ─────────────────────────────────────
ordem = []
for context, args in TRAIN_MAP.items():
    if context in yamls:
        n = contar_imagens(yamls[context])
        ordem.append((n, context, args))

ordem.sort(key=lambda x: x[0])

print("🔄 PIPELINE FEMHEALTH YOLOv8 (menor → maior)")
print("=" * 65)
for n, context, (nome, modelo, task, epochs) in ordem:
    print(f"  {context:<15} | {task:<10} | modelo: {modelo:<20} | ~{n} imgs treino")
print("=" * 65)

# ── Treina na ordem ───────────────────────────────────────────────────────
for n, context, (nome, modelo, task, epochs) in ordem:
    status[context] = safe_train(nome, modelo, yamls[context], task, epochs)

# ── Resumo ────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("🎯 RESUMO FINAL")
print(f"{'='*65}")
for context, st in status.items():
    path = resultados.get(context, "-")
    print(f"  {context:<15} | {st:<8} | {path}")

print("\n📌 INFERÊNCIA:")
for context, path in resultados.items():
    # Busca task pelo context diretamente no TRAIN_MAP
    _, _, task, _ = TRAIN_MAP.get(context, (None, None, "detect", None))
    pt = Path(path) / "weights" / "best.pt"
    if task == "classify":
        print(f"  {context}: YOLO('{pt}').predict(img, task='classify')")
    elif task == "pose":
        print(f"  {context}: YOLO('{pt}').predict(img, task='pose')")
    elif task == "segment":
        print(f"  {context}: YOLO('{pt}').predict(img, task='segment')")
    else:
        print(f"  {context}: YOLO('{pt}').predict(img)")



🔄 PIPELINE FEMHEALTH YOLOv8 (menor → maior)
  triagem         | pose       | modelo: yolov8n-pose.pt      | ~82 imgs treino
  cirurgia        | segment    | modelo: yolov8n-seg.pt       | ~948 imgs treino
  fisioterapia    | classify   | modelo: yolov8n-cls.pt       | ~1967 imgs treino
  consulta        | classify   | modelo: yolov8n-cls.pt       | ~3200 imgs treino

🚀 TREINO: AGGRESSIVE_POSES (pose)
📁 data: datasets\yolov8\triagem\data.yaml
New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.19  Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\yolov8\triagem\data.yaml, degrees=0.0, deterministic=True, device=0,

## 7. Avaliar Métricas dos Modelos

In [10]:
from ultralytics import YOLO

# Mapa inverso: nome_modelo → context
nome_para_context = {nome: ctx for ctx, (nome, _, _, _) in TRAIN_MAP.items()}

print(f"{'Contexto':<15} {'Task':<10} {'Métrica1':>10} {'Métrica2':>10} {'Métrica3':>10} {'Métrica4':>8}")
print('-' * 68)

for model_name, model_path in resultados.items():
    context = nome_para_context.get(model_name, model_name)
    yaml_path = yamls.get(context)
    if not yaml_path:
        print(f"{context:<15} ⚠️  yaml não encontrado")
        continue

    _, _, task, _ = TRAIN_MAP.get(context, (None, None, "detect", None))

    best_pt = Path(model_path) / "weights" / "best.pt"
    if not best_pt.exists():
        print(f"{context:<15} {task:<10} ⚠️  best.pt não encontrado em {best_pt}")
        continue
    model = YOLO(str(best_pt))

    try:
        if task == "detect":
            metrics = model.val(data=yaml_path, verbose=False, task=task)
            print(
                f"{context:<15} {task:<10}"
                f"{metrics.box.map50:>10.3f}"
                f"{metrics.box.map:>10.3f}"
                f"{metrics.box.mp:>10.3f}"
                f"{metrics.box.mr:>8.3f}"
            )

        elif task == "segment":
            metrics = model.val(data=yaml_path, verbose=False, task=task)
            print(
                f"{context:<15} {task:<10}"
                f"{metrics.seg.map50:>10.3f}"   # mAP50 segmentação
                f"{metrics.seg.map:>10.3f}"     # mAP50-95
                f"{metrics.box.map50:>10.3f}"   # mAP50 bbox
                f"{metrics.box.map:>8.3f}"      # mAP50-95 bbox
            )

        elif task == "pose":
            metrics = model.val(data=yaml_path, verbose=False, task=task)
            print(
                f"{context:<15} {task:<10}"
                f"{metrics.pose.map50:>10.3f}"  # mAP50 pose
                f"{metrics.pose.map:>10.3f}"    # mAP50-95
                f"{metrics.box.map50:>10.3f}"   # mAP50 bbox
                f"{metrics.box.map:>8.3f}"      # mAP50-95 bbox
            )

        elif task == "classify":
            metrics = model.val(data=str(Path(yaml_path).parent), verbose=False, task=task)
            print(
                f"{context:<15} {task:<10}"
                f"{metrics.top1:>10.3f}"        # top-1 accuracy
                f"{metrics.top5:>10.3f}"        # top-5 accuracy
                f"{'—':>10}"
                f"{'—':>8}"
            )

    except Exception as e:
        print(f"{context:<15} {task:<10} ❌ {e}")


Contexto        Task         Métrica1   Métrica2   Métrica3 Métrica4
--------------------------------------------------------------------
Ultralytics 8.4.19  Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
YOLOv8n-pose summary (fused): 82 layers, 3,289,964 parameters, 0 gradients, 9.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1150.7133.9 MB/s, size: 174.0 KB)
val: Scanning C:\Users\Xuanzang\dev\tech-challeng-fase-4\notebooks\datasets\yolov8\triagem\raw\labels\val.cache... 21 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21/21  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4s/it 2.9s8.5s
                   all         21         42      0.998          1      0.995      0.956      0.998          1      0.995      0.771
Speed: 2.6ms preprocess, 14.3ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Users\Xu

In [40]:
from pathlib import Path

src = Path("data/consulta")

# Mostra alguns pares facs ↔ png
facs_files = list(src.rglob("*_facs.txt"))[:5]
png_files  = list(src.rglob("*.png"))[:5]

print("FACS stems:")
for f in facs_files:
    print(f"  facs: {f.stem!r}  →  img esperada: {f.stem.replace('_facs', '')!r}")

print("\nPNG stems:")
for p in png_files:
    print(f"  png:  {p.stem!r}")


FACS stems:
  facs: 'll042t1aaaff001_facs'  →  img esperada: 'll042t1aaaff001'
  facs: 'll042t1aaaff002_facs'  →  img esperada: 'll042t1aaaff002'
  facs: 'll042t1aaaff003_facs'  →  img esperada: 'll042t1aaaff003'
  facs: 'll042t1aaaff004_facs'  →  img esperada: 'll042t1aaaff004'
  facs: 'll042t1aaaff005_facs'  →  img esperada: 'll042t1aaaff005'

PNG stems:
  png:  'll042t1aaaff001'
  png:  'll042t1aaaff002'
  png:  'll042t1aaaff003'
  png:  'll042t1aaaff004'
  png:  'll042t1aaaff005'


## 9. Resumo Final

In [15]:
import shutil
from pathlib import Path

best_pts = {
    "consulta":     r"C:\Users\Xuanzang\dev\tech-challeng-fase-4\runs\classify\models\yolov8_femhealth\classify\pain_detection\weights\best.pt",
    "fisioterapia": r"C:\Users\Xuanzang\dev\tech-challeng-fase-4\runs\classify\models\yolov8_femhealth\classify\rehab_postures\weights\best.pt",
    "triagem":      r"C:\Users\Xuanzang\dev\tech-challeng-fase-4\runs\pose\models\yolov8_femhealth\pose\aggressive_poses\weights\best.pt",
    "cirurgia":     r"C:\Users\Xuanzang\dev\tech-challeng-fase-4\runs\segment\models\yolov8_femhealth\segment\laparoscopia\weights\best.pt",
}

dest_dir = Path("models/yolov8_custom")
dest_dir.mkdir(parents=True, exist_ok=True)

print("\n📦 MODELOS TREINADOS:")
print("-" * 45)
for context, src in best_pts.items():
    dst = dest_dir / f"{context}.pt"
    shutil.copy2(src, dst)
    size_mb = dst.stat().st_size / 1e6
    print(f"  ✅ {context:<15} {dst} ({size_mb:.1f} MB)")

print("\n🚀 Próximo passo: uvicorn app.main:app --reload")



📦 MODELOS TREINADOS:
---------------------------------------------
  ✅ consulta        models\yolov8_custom\consulta.pt (3.0 MB)
  ✅ fisioterapia    models\yolov8_custom\fisioterapia.pt (3.0 MB)
  ✅ triagem         models\yolov8_custom\triagem.pt (6.8 MB)
  ✅ cirurgia        models\yolov8_custom\cirurgia.pt (6.8 MB)

🚀 Próximo passo: uvicorn app.main:app --reload
